# Data Cleaning — `Tourist_Accommodation` (revisión 13/07/2026)

Revisión y actualización del notebook de limpieza `Data_Cleaning_06_07_2026.ipynb`, a partir del
nuevo fichero de datos crudos disponible (`raw_dataset_06_07_2026_1.csv`).

**Motivo de la revisión:** el fichero crudo actual incluye columnas que la semana pasada no
existían en el origen (`occupancy_30/60/90/365`, `occupancy_rate_30/60/90/365`, `rating_above_80`,
`is_instant_bookable_numeric`), y el formato de `insert_date` cambió de `dd/mm/aaaa` a `aaaa-mm-dd`.
Se mantiene toda la lógica de limpieza ya validada la semana pasada y **solo se añade/ajusta lo
estrictamente necesario** para cubrir estos cambios y las variables de interés de los tres
departamentos (ver `Preguntas_de_negocio_por_departamento_13_07_2026.docx`).

**Puntos que se cubren (según instrucciones de Data Cleaning):**
1. Corrección de tipos de datos
2. Eliminación o corrección de duplicados
3. Tratamiento de valores faltantes
4. Validación y corrección de valores atípicos
5. Estandarización de formatos

**Salida final:** un único CSV limpio (`../Data/clean_dataset_13_07_2026.csv`) sobre el que se
aplicará la fase de *Data Transformation*.

> **Alcance de esta fase.** Aquí solo se limpia y se corrigen tipos/formatos. **NO** se hacen aquí
> (van en *Transformation*): creación de variables derivadas de negocio, codificación categórica
> definitiva, ni reducción de dimensionalidad.
> No se elimina información salvo lo estrictamente necesario (versiones antiguas de un mismo
> alojamiento).

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)

In [2]:
# Nombre del fichero: única línea a cambiar cuando llegue un dataset nuevo
DATASET_FILE = 'raw_dataset_06_07_2026_1.csv'

def encontrar_raiz_proyecto(nombre_carpeta='Equip_34'):
    '''
    Función para encontrar la carpeta raíz del proyecto subiendo desde el directorio actual.
    '''
    actual = Path.cwd()
    for carpeta in [actual] + list(actual.parents):
        if carpeta.name == nombre_carpeta:
            return carpeta
    raise FileNotFoundError(f"No se encontró la carpeta '{nombre_carpeta}' subiendo desde {actual}")

raiz_proyecto = encontrar_raiz_proyecto('Equip_34')
ruta = raiz_proyecto / 'Data' / DATASET_FILE

print(f"Ruta resuelta: {ruta}")
df_original = pd.read_csv(ruta)
df = df_original.copy()

print("Dimensiones del crudo:", df.shape)

Ruta resuelta: /home/claude/work/Equip_34/Data/raw_dataset_06_07_2026_1.csv


Dimensiones del crudo: (8000, 46)


In [3]:
# Copia de trabajo para no alterar el DataFrame original importado
df_clean = df.copy()

## 0. Columnas nuevas respecto al 06/07

El fichero crudo actual ya trae, calculadas en origen, las columnas de ocupación
(`occupancy_30/60/90/365`, `occupancy_rate_30/60/90/365`), `rating_above_80` y
`is_instant_bookable_numeric`. Se verifica su coherencia antes de seguir con la limpieza
habitual, ya que **`occupancy_365` es la variable objetivo acordada con Verónica** para la
pregunta de negocio de operaciones (habitaciones/baños/camas vs. ocupación), en sustitución
del enfoque inicial en "disponibilidad".

In [4]:
# occupancy_x + availability_x debe ser siempre igual al plazo (30/60/90/365)
for periodo in [30, 60, 90, 365]:
    suma = df_clean[f'occupancy_{periodo}'] + df_clean[f'availability_{periodo}']
    print(f"occupancy_{periodo} + availability_{periodo} -> valores únicos: {suma.unique()}")

# occupancy_rate_x debe ser coherente con occupancy_x / periodo * 100
print()
for periodo in [30, 60, 90, 365]:
    calc = (df_clean[f'occupancy_{periodo}'] / periodo * 100).round(2)
    diff = (calc - df_clean[f'occupancy_rate_{periodo}']).abs()
    print(f"occupancy_rate_{periodo}: diferencia máxima vs. cálculo propio = {diff.max()}")

print("\nNulos en columnas de ocupación:")
print(df_clean[[f'occupancy_{p}' for p in [30,60,90,365]]].isna().sum())

occupancy_30 + availability_30 -> valores únicos: [30]
occupancy_60 + availability_60 -> valores únicos: [60]
occupancy_90 + availability_90 -> valores únicos: [90]
occupancy_365 + availability_365 -> valores únicos: [365]

occupancy_rate_30: diferencia máxima vs. cálculo propio = 0.0
occupancy_rate_60: diferencia máxima vs. cálculo propio = 0.0
occupancy_rate_90: diferencia máxima vs. cálculo propio = 0.0
occupancy_rate_365: diferencia máxima vs. cálculo propio = 0.0

Nulos en columnas de ocupación:
occupancy_30     0
occupancy_60     0
occupancy_90     0
occupancy_365    0
dtype: int64


In [5]:
# is_instant_bookable_numeric: debe coincidir con el mapeo de is_instant_bookable
chk = df_clean['is_instant_bookable'].map({'VERDADERO': 1, 'FALSO': 0})
print("is_instant_bookable_numeric, discrepancias:", (chk != df_clean['is_instant_bookable_numeric']).sum())
print(df_clean['is_instant_bookable'].value_counts(dropna=False))
print(df_clean['is_instant_bookable_numeric'].value_counts(dropna=False))

is_instant_bookable_numeric, discrepancias: 0
is_instant_bookable
VERDADERO    4439
FALSO        3561
Name: count, dtype: int64
is_instant_bookable_numeric
1    4439
0    3561
Name: count, dtype: int64


Ambos bloques de columnas llegan ya coherentes y completos (0 nulos, 0 discrepancias):
no requieren imputación ni corrección, solo su tipo de dato definitivo (sección 1).

## 1. Corrección de tipos de datos

- `bathrooms`, `bedrooms` y `beds` llegan como `float64` aunque son recuentos → a entero
  nullable (`Int64`, admite nulos).
- `first_review_date`, `last_review_date` e `insert_date` son fechas en texto, pero **no
  siempre en el mismo formato**: puede ser `dd/mm/aaaa` o `aaaa-mm-dd` según la columna, y
  según nos recuerda Data Understanding, podría no ser siempre el mismo dentro de la propia
  columna en próximas entregas del dataset. Por eso se usa una función que detecta el formato
  de cada valor y lo convierte a `datetime` en lugar de asumir un único formato fijo (así el
  notebook sigue funcionando aunque el formato cambie de nuevo).
- `rating_above_80` llega como texto (`'True'`/`'False'`) → a booleano nullable (`boolean`).

In [6]:
# --- Recuentos que venían como texto/float -> entero nullable (Int64) ---
count_cols = ['bathrooms', 'bedrooms', 'beds']
for col in count_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').astype('Int64')

# --- Fechas: admite dd/mm/aaaa o aaaa-mm-dd, detectando el formato de cada valor ---
def convertir_fecha(serie):
    '''Convierte una columna de fechas en texto a datetime, admitiendo
    los formatos dd/mm/aaaa y aaaa-mm-dd (pueden convivir en el dataset).'''
    texto = serie.astype('string')
    es_iso = texto.str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)

    resultado = pd.Series(pd.NaT, index=serie.index, dtype='datetime64[us]')
    resultado.loc[es_iso] = pd.to_datetime(texto.loc[es_iso], format='%Y-%m-%d', errors='coerce')
    resultado.loc[~es_iso] = pd.to_datetime(texto.loc[~es_iso], format='%d/%m/%Y', errors='coerce')
    return resultado

date_cols = ['first_review_date', 'last_review_date', 'insert_date']
for col in date_cols:
    df_clean[col] = convertir_fecha(df_clean[col])

df_clean[count_cols + date_cols].dtypes

bathrooms                     Int64
bedrooms                      Int64
beds                          Int64
first_review_date    datetime64[us]
last_review_date     datetime64[us]
insert_date          datetime64[us]
dtype: object

In [7]:
# Comprobación: la conversión no debe crear nulos "nuevos" inesperados
print("Nulos tras convertir tipos (deben coincidir con los originales):")
for col in count_cols + date_cols:
    print(f"  {col:20s} nulos: {df_clean[col].isna().sum():5d}  |  original: {df[col].isna().sum()}")

Nulos tras convertir tipos (deben coincidir con los originales):
  bathrooms            nulos:    43  |  original: 43
  bedrooms             nulos:    39  |  original: 39
  beds                 nulos:     8  |  original: 8
  first_review_date    nulos:  1614  |  original: 1614
  last_review_date     nulos:  1615  |  original: 1615
  insert_date          nulos:     0  |  original: 0


In [8]:
# rating_above_80: texto 'True'/'False' -> booleano nullable
df_clean['rating_above_80'] = (
    df_clean['rating_above_80']
    .map({'True': True, 'False': False, True: True, False: False})
    .astype('boolean')
)
print(df_clean['rating_above_80'].dtype)
print(df_clean['rating_above_80'].value_counts(dropna=False))

boolean
rating_above_80
True     5622
<NA>     1696
False     682
Name: count, dtype: Int64


## 2. Eliminación o corrección de duplicados

No hay filas completamente idénticas, pero sí `apartment_id` repetidos: el mismo alojamiento
volcado en fechas distintas. Se conserva únicamente la **última foto** (máximo `insert_date`) de
cada `apartment_id`. La lógica es idéntica a la semana pasada; el resultado (307 registros
eliminados) coincide porque el contenido de fondo es el mismo, solo cambia el formato de
`insert_date`, ya corregido en la sección anterior.

In [9]:
# Filas 100% duplicadas
print("Filas completamente duplicadas:", df_clean.duplicated().sum())

# apartment_id repetidos (mismo alojamiento en distintos volcados)
rep = df_clean['apartment_id'].value_counts()
rep = rep[rep > 1]
print(f"apartment_id repetidos: {rep.shape[0]}  (filas implicadas: {int(rep.sum())})")
print("Fecha global mas reciente de volcado (insert_date):", df_clean['insert_date'].max().date())

# Nos quedamos con el registro de mayor insert_date por apartment_id
antes = len(df_clean)
df_clean = (
    df_clean.sort_values('insert_date')
            .drop_duplicates(subset='apartment_id', keep='last')
            .reset_index(drop=True)
)
print(f"\nRegistros eliminados (versiones antiguas): {antes - len(df_clean)}")
print("Dimensiones tras deduplicar:", df_clean.shape)
print("apartment_id unico ahora?", df_clean['apartment_id'].is_unique)

Filas completamente duplicadas: 0
apartment_id repetidos: 299  (filas implicadas: 606)
Fecha global mas reciente de volcado (insert_date): 2021-02-27

Registros eliminados (versiones antiguas): 307
Dimensiones tras deduplicar: (7693, 46)
apartment_id unico ahora? True


## 3. Tratamiento de valores faltantes

Se **conservan** los registros con nulos (no se eliminan) y **no se imputan** valores que
puedan sesgar los análisis posteriores:

- `review_scores_*`, `rating_above_80`, `first_review_date`, `last_review_date`,
  `reviews_per_month`: nulos **estructurales** = alojamientos sin reseñas
  (`number_of_reviews = 0`). Se dejan como nulos.
- `has_availability`: solo `VERDADERO` + nulos (no hay `FALSO`); los nulos se dejan como
  desconocido (no se convierten a falso). Se crea `has_availability_numeric` con valores 1 y 0
  respetando los nulos (esta columna ya llegaba calculada en el crudo: se verifica que coincide
  con el mapeo directo antes de darla por buena).
- `neighbourhood_district`, `price`, `bathrooms`, `bedrooms`, `beds`, `name`, `description`:
  se conservan sin imputar.
- `occupancy_*` / `occupancy_rate_*` / `is_instant_bookable_numeric`: sin nulos (verificado en
  la sección 0), no requieren tratamiento.

In [10]:
# Panorama de nulos tras deduplicar
nulos = pd.DataFrame({
    "nulos": df_clean.isnull().sum(),
    "%": (df_clean.isnull().mean() * 100).round(2)
})
nulos[nulos["nulos"] > 0].sort_values("%", ascending=False)

,nulos,%
neighbourhood_district,3024,39.31
review_scores_location,1649,21.44
review_scores_value,1649,21.44
review_scores_checkin,1648,21.42
review_scores_accuracy,1643,21.36
review_scores_communication,1639,21.31
review_scores_cleanliness,1637,21.28
rating_above_80,1634,21.24
review_scores_rating,1634,21.24
last_review_date,1555,20.21


In [11]:
# Los nulos de valoraciones son estructurales: alojamientos sin reseñas
sin_reviews = df_clean['number_of_reviews'] == 0
print("Alojamientos con number_of_reviews == 0 :", sin_reviews.sum())
print("  de ellos con review_scores_rating nulo:", (sin_reviews & df_clean['review_scores_rating'].isna()).sum())
print("Con reseñas (>0) pero rating nulo (nulo real, no estructural):",
      ((~sin_reviews) & df_clean['review_scores_rating'].isna()).sum())
print("rating_above_80 nulo coincide exactamente con review_scores_rating nulo:",
      (df_clean['rating_above_80'].isna() == df_clean['review_scores_rating'].isna()).all())

print("\nhas_availability (solo VERDADERO + nulos, sin FALSO):")
print(df_clean['has_availability'].value_counts(dropna=False))

Alojamientos con number_of_reviews == 0 : 1551
  de ellos con review_scores_rating nulo: 1551
Con reseñas (>0) pero rating nulo (nulo real, no estructural): 83
rating_above_80 nulo coincide exactamente con review_scores_rating nulo: True

has_availability (solo VERDADERO + nulos, sin FALSO):
has_availability
VERDADERO    7159
NaN           534
Name: count, dtype: int64


In [12]:
# Verificamos que has_availability_numeric (ya venía en el crudo) coincide con el mapeo directo
mapeo_directo = df_clean['has_availability'].map({'VERDADERO': 1, 'FALSO': 0}).astype('Int64')
discrepancias = (mapeo_directo != df_clean['has_availability_numeric'])
discrepancias = discrepancias & ~(mapeo_directo.isna() & df_clean['has_availability_numeric'].isna())
print("Discrepancias has_availability_numeric vs mapeo directo:", int(discrepancias.sum()))

print(df_clean['has_availability_numeric'].value_counts(dropna=False))
df_clean[['has_availability', 'has_availability_numeric']].tail()

Discrepancias has_availability_numeric vs mapeo directo: 0
has_availability_numeric
1.0    7159
NaN     534
Name: count, dtype: int64


,has_availability,has_availability_numeric
7688,VERDADERO,1.0
7689,VERDADERO,1.0
7690,VERDADERO,1.0
7691,VERDADERO,1.0
7692,VERDADERO,1.0


In [13]:
df_clean["reviews_per_month"].describe()

count    6139.000000
mean      124.895586
std       154.556071
min         1.000000
25%        17.000000
50%        58.000000
75%       180.000000
max      1273.000000
Name: reviews_per_month, dtype: float64

In [14]:
df_clean["reviews_per_month"].sort_values(ascending=False).head(10)

4703    1273.0
2367    1213.0
6054    1125.0
6217    1045.0
4251     952.0
995      938.0
4134     914.0
4122     911.0
3522     868.0
6387     868.0
Name: reviews_per_month, dtype: float64

## 4. Validación y corrección de valores atípicos

Confirmamos que los valores extremos son plausibles (no errores) y los **mantenemos**. Solo se
revisa coherencia; no se elimina nada.

> `reviews_per_month` presenta valores muy altos (máx ≈ 1273). Parece estar escalado (×100)
> respecto al valor mensual real; **no se corrige aquí**, se revisará en *Transformation*.

> Las columnas de ocupación (`occupancy_365`, etc.) ya se validaron en la sección 0: son
> complementarias exactas de `availability_x` sobre el mismo plazo, así que no pueden tener
> valores fuera de rango (`[0, plazo]`) por construcción.

In [15]:
num_check = ['price', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
             'minimum_nights', 'maximum_nights', 'number_of_reviews', 'reviews_per_month']
df_clean[num_check].describe().round(2)

,price,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,number_of_reviews,reviews_per_month
count,7534.00,7693.00,7652.0,7654.0,7685.0,7693.00,7693.00,7693.00,6139.00
mean,1005.60,4.32,1.6,1.95,2.97,4.58,759.40,31.39,124.90
std,842.75,2.60,0.98,1.29,2.3,11.96,498.53,57.35,154.56
min,60.00,1.00,0.0,0.0,0.0,1.00,1.00,0.00,1.00
25%,450.00,2.00,1.0,1.0,1.0,1.00,61.00,1.00,17.00
50%,750.00,4.00,1.0,2.0,2.0,2.00,1125.00,8.00,58.00
75%,1220.00,6.00,2.0,3.0,4.0,4.00,1125.00,36.00,180.00
max,6071.00,29.00,12.0,16.0,30.0,365.00,1125.00,588.00,1273.00


In [16]:
# Recuento de outliers por IQR (solo informativo: NO se eliminan)
def n_outliers_iqr(s):
    s = s.dropna().astype(float)
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < low) | (s > high)).sum())

resumen_out = pd.Series({c: n_outliers_iqr(df_clean[c]) for c in num_check}, name='outliers_IQR')
print(resumen_out.to_string())

# Casos limite coherentes que SE CONSERVAN (pueden aportar info en otras columnas)
print("\nAlojamientos con beds == 0      :", int((df_clean['beds'] == 0).sum()), "(se conservan)")
print("Alojamientos con bathrooms == 0 :", int((df_clean['bathrooms'] == 0).sum()), "(se conservan)")
print("price -> min:", df_clean['price'].min(), "| max:", df_clean['price'].max(), "| nulos:", int(df_clean['price'].isna().sum()))

price                572
accommodates          63
bathrooms            361
bedrooms              37
beds                 178
minimum_nights       431
maximum_nights         0
number_of_reviews    797
reviews_per_month    384

Alojamientos con beds == 0      : 80 (se conservan)
Alojamientos con bathrooms == 0 : 14 (se conservan)
price -> min: 60.0 | max: 6071.0 | nulos: 159


In [17]:
# occupancy_365 fuera de rango [0, 365]? (comprobación de sanidad para la pregunta de operaciones)
fuera_rango = ((df_clean['occupancy_365'] < 0) | (df_clean['occupancy_365'] > 365)).sum()
print("occupancy_365 fuera de [0, 365]:", fuera_rango)
df_clean['occupancy_365'].describe()

occupancy_365 fuera de [0, 365]: 0


count    7693.000000
mean      177.297803
std       129.894312
min         0.000000
25%        49.000000
50%       177.000000
75%       299.000000
max       365.000000
Name: occupancy_365, dtype: float64

## 5. Estandarización de formatos

- Espacios sobrantes en columnas de texto.
- `price` → `price_€` (se mantiene numérico, solo cambia el nombre para dejar explícita la
  unidad, como en la semana pasada).
- `city`: **en el fichero nuevo llega en minúsculas** (`barcelona`, `madrid`...) en vez de
  mixto como antes; se normaliza igual que la semana pasada (primera letra en mayúscula), y el
  resultado es equivalente.
- Limpieza de `amenities_list` (codificación, sinónimos, tokens de traducción faltante).

In [18]:
# Espacios sobrantes en columnas de texto
text_cols = df_clean.select_dtypes(include=['object']).columns
for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

# price -> price_€ (se mantiene numerico)
df_clean = df_clean.rename(columns={'price': 'price_€'})
print("Columna de precio:", [c for c in df_clean.columns if 'price' in c], "| dtype:", df_clean['price_€'].dtype)

# Columna `city` pasa a ser mayuscula la primera letra
print("Valores de city antes de normalizar:", sorted(df_clean['city'].str.strip().unique()))
df_clean["city"] = df_clean["city"].str.strip().str.capitalize()
print("\ncolumna City normalizada:")
print(df_clean['city'].value_counts())

Columna de precio: ['price_€'] | dtype: float64
Valores de city antes de normalizar: ['barcelona', 'girona', 'madrid', 'malaga', 'mallorca', 'menorca', 'sevilla', 'valencia']

columna City normalizada:
city
Barcelona    2261
Madrid       1645
Mallorca     1265
Girona       1198
Sevilla       403
Malaga        394
Valencia      360
Menorca       167
Name: count, dtype: int64


/tmp/ipykernel_542/110552629.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df_clean.select_dtypes(include=['object']).columns


In [19]:
# Cuantificacion del problema de codificacion (informativo, no se altera el dato)
mojibake_cols = ['name', 'neighbourhood_name', 'description']
for col in mojibake_cols:
    n = df_clean[col].fillna('').str.contains('�').sum()
    print(f"{col:20s}: {n} registros con caracter de codificacion perdido (acento perdido en origen)")

name                : 1569 registros con caracter de codificacion perdido (acento perdido en origen)
neighbourhood_name  : 1681 registros con caracter de codificacion perdido (acento perdido en origen)


description         : 5269 registros con caracter de codificacion perdido (acento perdido en origen)


In [20]:
# Tratamiento de translation missing: en.hosting_amenity_49/50
print(
    "Registros afectados:",
    df_clean["amenities_list"].str.contains(
        "translation missing",
        case=False,
        na=False
    ).sum()
)

Registros afectados: 927


In [21]:
# identificamos la existencia de los valores erroneos
(df_clean.loc[
        df_clean["amenities_list"].str.contains(
            "translation missing",
            case=False,
            na=False
        ),
        "amenities_list"
    ]
    .str.extractall(r"(translation missing:[^,\]]+)")
    .value_counts())

0                                         
translation missing: en.hosting_amenity_50    841
translation missing: en.hosting_amenity_49    633
Name: count, dtype: int64

In [22]:
# eliminamos los valores erroneos
valores_eliminar = [
    "translation missing: en.hosting_amenity_49",
    "translation missing: en.hosting_amenity_50"
]

for token in valores_eliminar:
    df_clean["amenities_list"] = (
        df_clean["amenities_list"]
        .str.replace(token, "", regex=False)
    )

In [23]:
#eliminamos posibles , dobles que puedan quedar de la eliminacion
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
        .str.replace(r"\s*,\s*,", ",", regex=True)
        .str.replace(r"^,\s*", "", regex=True)
        .str.replace(r",\s*$", "", regex=True)
        .str.replace(r"\s{2,}", " ", regex=True)
        .str.strip()
)

In [24]:
#en los casos en que quede vacia la lista, se convierten a nulos
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
        .replace(r"^\s*$", pd.NA, regex=True)
)

In [25]:
amenities = (
    df_clean["amenities_list"]
        .dropna()
        .str.split(",")
        .explode()
        .str.strip()
)

In [26]:
# confirmamos que no hay registros vacios
print("Amenities vacías:", (amenities == "").sum())
print("Amenities únicas:", amenities.nunique())

Amenities vacías: 295
Amenities únicas: 321


In [27]:
# Unificación del nombre de ciudad (por si quedara alguna variante suelta)
df_clean["city"] = df_clean["city"].replace({
    "Malaga": "Málaga"
})

Se detectaron registros donde la columna `amenities_list` contenía una cadena vacía
(`''`) en lugar de un valor nulo. Se estandarizaron a `NaN` para representar la ausencia de
información y evitar generar una comodidad vacía durante la separación de la lista.

### Estandarización de sinónimos y codificación en `amenities_list`

In [28]:
# Obtener todas las amenities únicas
amenities = (
    df_clean["amenities_list"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

amenities.sort_values().unique()

<StringArray>
[                            '',             '24-hour check-in',
                      '40 HDTV',         '43 HDTV with Netflix',
        'Accessible-height bed',     'Accessible-height toilet',
             'Air conditioning',            'Air conditioning]',
                  'Amazon Echo',                    'BBQ grill',
 ...
                         'Wifi',          'Wifi u2013 100 Mbps',
                        'Wifi]',                'Window guards',
               'Window guards]',                 'Wine glasses',
            'Wireless Internet',                            ']',
 'smooth pathway to front door',                       'toilet']
Length: 321, dtype: str

In [29]:
sinonimos = {

    # Sinonimos
    "Smoke detector": "Smoke alarm",
    "Carbon monoxide detector": "Carbon monoxide alarm",
    "Wireless Internet": "Wifi",

    # Capitalizacion
    "Self Check-In": "Self check-in",
    "Laptop-friendly workspace": "Laptop friendly workspace",
    "Doorman Entry": "Doorman",

    # Problemas de codificacion
    "Children\\u2019s books and toys": "Children\'s books and toys",
    "Children\ufffds books and toys": "Children\'s books and toys",
    "Childrenu2019s books and toys": "Children\'s books and toys",

    "Children\\u2019s dinnerware": "Children\'s dinnerware",
    "Children\ufffds dinnerware": "Children\'s dinnerware",
    "Childrenu2019s dinnerware": "Children\'s dinnerware",

    "Pack \\u2019n Play/travel crib": "Pack \'n Play/travel crib",
    "Pack \ufffdn Play/travel crib": "Pack \'n Play/travel crib",
    "Pack u2019n Play/travel crib": "Pack \'n Play/travel crib",

    "Wifi u2013 100 Mbps": "Wifi",
    "Washer u2013 In unit": "Washer",
    "Washer u2013u00a0In unit": "Washer",

    "Free driveway parking on premises u2013 1 space":
    "Free driveway parking on premises - 1 space",

    # Errores puntuales
    "43 HDTVwith Netflix": "43 HDTV with Netflix",
    "Pour Over Coffee": "Pour-over coffee",
    "Ski in/Ski out": "Ski-in/Ski-out",
    "smooth pathway to front door": "Smooth pathway to front door",
    "toilet": "Toilet"
}

for viejo, nuevo in sinonimos.items():
    df_clean["amenities_list"] = (
        df_clean["amenities_list"]
        .str.replace(viejo, nuevo, regex=False)
    )

In [30]:
# Eliminar espacios alrededor de las comas
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
    .str.replace(r"\s*,\s*", ",", regex=True)
)

# Eliminar comas repetidas
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
    .str.replace(r",\s*,+", ",", regex=True)
)

# Eliminar coma final
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
    .str.replace(r",\s*$", "", regex=True)
    .str.strip()
)

In [31]:
amenities = (
    df_clean["amenities_list"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

# No deben quedar elementos vacios
print("Amenities vacías:", (amenities == "").sum())

# Numero de amenities distintas
print("Amenities únicas:", amenities.nunique())

# Frecuencias
amenities_df = (
    amenities
    .value_counts()
    .reset_index()
)

amenities_df.columns = ["Amenity", "Frecuencia"]

amenities_df

Amenities vacías: 0
Amenities únicas: 307


,Amenity,Frecuencia
0,Kitchen,7055
1,Wifi,6959
2,Essentials,6823
3,Washer,6489
4,TV,6456
...,...,...
302,Outdoor dining area,1
303,Drying rack for clothing,1
304,Outdoor shower,1
305,Dining table,1


In [32]:
#convertimos las cadenas vacias a nulos
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
    .replace(r"^\s*$", pd.NA, regex=True)
)

In [33]:
amenities = (
    df_clean["amenities_list"]
        .dropna()
        .str.split(",")
        .explode()
        .str.strip()
)

In [34]:
# confirmamos que no hay registros vacios
print("Amenities vacías:", (amenities == "").sum())
print("Amenities únicas:", amenities.nunique())

Amenities vacías: 0
Amenities únicas: 307


In [35]:
# limpieza de formato. eliminacion de `]`
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
    .str.replace("]", "", regex=False)
)

Se detectaron registros donde la columna `amenities_list` contenía una cadena vacía
(`''`) tras la limpieza de tokens y sinónimos. Se estandarizaron a `NaN` para representar la
ausencia de información y evitar generar una comodidad vacía durante la separación de la
lista.

## 6. Verificación final y variables de interés por departamento

Antes de exportar, se comprueba que están presentes las variables necesarias para las tres
preguntas de negocio (`Preguntas_de_negocio_por_departamento_13_07_2026.docx`):

| Departamento | Variables necesarias | ¿Presentes? |
|---|---|---|
| Marketing y comunicación | `city`, `neighbourhood_name`, `review_scores_location`, `minimum_nights`, `maximum_nights` | Sí |
| **Operaciones e inventario (Giorgia)** | `bedrooms`, `bathrooms`, `beds`, `occupancy_365` (y `occupancy_30/60/90`), `is_instant_bookable`, `city` | Sí |
| Experiencia del cliente | `price_€`, `review_scores_rating`, `city` | Sí |

Se genera un **único CSV** con el resultado de la limpieza, listo para la fase de
*Data Transformation*.

In [36]:
print("Dimensiones finales:", df_clean.shape)
print("apartment_id unico :", df_clean['apartment_id'].is_unique)
print("Filas duplicadas   :", df_clean.duplicated().sum())

variables_interes = {
    "Marketing": ['city', 'neighbourhood_name', 'review_scores_location',
                  'minimum_nights', 'maximum_nights'],
    "Operaciones (Giorgia)": ['bedrooms', 'bathrooms', 'beds', 'occupancy_30',
                               'occupancy_60', 'occupancy_90', 'occupancy_365',
                               'is_instant_bookable', 'city'],
    "Experiencia cliente": ['price_€', 'review_scores_rating', 'city'],
}
print("\nComprobación de variables de interés por departamento:")
for depto, cols in variables_interes.items():
    faltan = [c for c in cols if c not in df_clean.columns]
    print(f"  {depto:25s} -> faltan: {faltan if faltan else 'ninguna'}")

print("\nTipos de datos finales:")
df_clean.info()

Dimensiones finales: (7693, 46)
apartment_id unico : True
Filas duplicadas   : 0

Comprobación de variables de interés por departamento:
  Marketing                 -> faltan: ninguna
  Operaciones (Giorgia)     -> faltan: ninguna
  Experiencia cliente       -> faltan: ninguna

Tipos de datos finales:
<class 'pandas.DataFrame'>
RangeIndex: 7693 entries, 0 to 7692
Data columns (total 46 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7693 non-null   int64         
 1   name                         7690 non-null   str           
 2   description                  7643 non-null   str           
 3   host_id                      7693 non-null   int64         
 4   neighbourhood_name           7693 non-null   str           
 5   neighbourhood_district       4669 non-null   str           
 6   room_type                    7693 non-null   str           
 7   accommodate

In [37]:
# Exportacion del CSV final (una sola tabla limpia) para Data Transformation
output_dir = Path('..') / 'Data'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'clean_dataset_13_07_2026.csv'

# utf-8 (sin BOM) para un round-trip limpio con pandas en la fase siguiente
df_clean.to_csv(output_path, index=False, encoding='utf-8')

print("CSV limpio guardado en:", output_path.resolve())
print("Filas x columnas:", df_clean.shape)

CSV limpio guardado en: /home/claude/work/Equip_34/Data/clean_dataset_13_07_2026.csv
Filas x columnas: (7693, 46)
